1. Selection and filtering

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import pdfplumber


In [2]:
pdf_path="/home/paragchengappa/programming/UPI-Spending-Analyser/data/raw/bankstats.pdf"
text=""   # called an accumulator, stores all text

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        page_text=page.extract_text()
        if page_text:
            text+=page_text+ "\n"         
print(text[:2000])


Transaction statement
9019517497, paragchengappa@gmail.com
Transactionstatementperiod Sent Received
01January2026-30June2026 ₹77,368.99 ₹70,536
Date&time Transactiondetails Amount
03Jan,2026 PaidtoJAYARAMAPB ₹30
07:41AM UPITransactionID:600358945338
PaidbyStateBankofIndia7202
03Jan,2026 PaidtoSHIVANNA SHIVANNA ₹231
02:17PM UPITransactionID:636962750553
PaidbyStateBankofIndia7202
05Jan,2026 PaidtoPARASHURAMA ₹56.15
07:29AM UPITransactionID:600553332781
PaidbyStateBankofIndia7202
05Jan,2026 PaidtoRAMESHMR ₹5
09:45AM UPITransactionID:600558105853
PaidbyStateBankofIndia7202
05Jan,2026 ReceivedfromMC NALINAKSHI ₹1,100
01:14PM UPITransactionID:969952655934
PaidtoStateBankofIndia7202
05Jan,2026 ReceivedfromMC NALINAKSHI ₹300
01:19PM UPITransactionID:527376183990
PaidtoStateBankofIndia7202
05Jan,2026 PaidtoBlinkit ₹1,112
01:19PM UPITransactionID:637121529011
PaidbyStateBankofIndia7202
06Jan,2026 PaidtoBMTCBUSKA57F1114 ₹15
12:03PM UPITransactionID:600608819250
PaidbyStateBankofIndia7202
Note:Th

In [3]:
transactions=[]

In [4]:
lines=text.splitlines()

In [5]:
print(lines)

['Transaction statement', '9019517497, paragchengappa@gmail.com', 'Transactionstatementperiod Sent Received', '01January2026-30June2026 ₹77,368.99 ₹70,536', 'Date&time Transactiondetails Amount', '03Jan,2026 PaidtoJAYARAMAPB ₹30', '07:41AM UPITransactionID:600358945338', 'PaidbyStateBankofIndia7202', '03Jan,2026 PaidtoSHIVANNA SHIVANNA ₹231', '02:17PM UPITransactionID:636962750553', 'PaidbyStateBankofIndia7202', '05Jan,2026 PaidtoPARASHURAMA ₹56.15', '07:29AM UPITransactionID:600553332781', 'PaidbyStateBankofIndia7202', '05Jan,2026 PaidtoRAMESHMR ₹5', '09:45AM UPITransactionID:600558105853', 'PaidbyStateBankofIndia7202', '05Jan,2026 ReceivedfromMC NALINAKSHI ₹1,100', '01:14PM UPITransactionID:969952655934', 'PaidtoStateBankofIndia7202', '05Jan,2026 ReceivedfromMC NALINAKSHI ₹300', '01:19PM UPITransactionID:527376183990', 'PaidtoStateBankofIndia7202', '05Jan,2026 PaidtoBlinkit ₹1,112', '01:19PM UPITransactionID:637121529011', 'PaidbyStateBankofIndia7202', '06Jan,2026 PaidtoBMTCBUSKA57F1

In [6]:
import re
i=0

while i<len(lines):
    line=lines[i]

    if re.match(r"\d{2}[A-Za-z]{3,9},\d{4}",line):
        line1=lines[i]
        line2=lines[i+1]
        line3=lines[i+2]

        date=re.search(r"\d{2}[A-Za-z]{3,9},\d{4}",line1).group()

        amount=re.search(r"₹[\d,\.]+",line1).group()

        details=line1.replace(date,"")             #details=string after replacing date w "" from line1
        details=details.replace(amount,"").strip() #details=replace amount from DETAILS w ""
                                                   #strip blank spaces "","" which replaced date and amount
                                                   #and then assign to details

        time=re.search(r"\d{2}:\d{2}[AP]M",line2).group()

        upi=re.search(r"\d{12}",line2).group()
        bank=line3                                 # since in text, third line of each transaction is Paidby/toStateBankOfIndia

        transactions.append([date,time,details,amount,upi,bank])
        i+=3
    else:
        i+=1
                


In [7]:
df=pd.DataFrame(transactions,columns=["Date","Time","Details","Amount","UPI_ID","Bank"])
df.head()

,Date,Time,Details,Amount,UPI_ID,Bank
0,"03Jan,2026",07:41AM,PaidtoJAYARAMAPB,₹30,600358945338,PaidbyStateBankofIndia7202
1,"03Jan,2026",02:17PM,PaidtoSHIVANNA SHIVANNA,₹231,636962750553,PaidbyStateBankofIndia7202
2,"05Jan,2026",07:29AM,PaidtoPARASHURAMA,₹56.15,600553332781,PaidbyStateBankofIndia7202
3,"05Jan,2026",09:45AM,PaidtoRAMESHMR,₹5,600558105853,PaidbyStateBankofIndia7202
4,"05Jan,2026",01:14PM,ReceivedfromMC NALINAKSHI,"₹1,100",969952655934,PaidtoStateBankofIndia7202


In [8]:
df["Date"]=pd.to_datetime(df["Date"],format= "%d%b,%Y")
df["Amount"]=(df["Amount"].str.replace("₹","",regex=False)
              .str.replace(",","",regex=False))
df["Amount"]=df["Amount"].astype(float)
df.info
df.isnull().sum()
df.duplicated().sum()

0

In [9]:
print(df["Details"])

0               PaidtoJAYARAMAPB
1        PaidtoSHIVANNA SHIVANNA
2              PaidtoPARASHURAMA
3                 PaidtoRAMESHMR
4      ReceivedfromMC NALINAKSHI
                 ...            
508            PaidtoAmazonIndia
509    ReceivedfromMC NALINAKSHI
510                 PaidtoMyntra
511    ReceivedfromMC NALINAKSHI
512                PaidtoBlinkit
Name: Details, Length: 513, dtype: object


In [10]:
df["Transaction_Type"]=df["Details"].str.contains(r"Paid",case=False, na=False).map({True:"Paid",
                                                                                     False:"Received"
                                                                                    })

In [11]:
#DEL
print(df["Transaction_Type"])

0          Paid
1          Paid
2          Paid
3          Paid
4      Received
         ...   
508        Paid
509    Received
510        Paid
511    Received
512        Paid
Name: Transaction_Type, Length: 513, dtype: object


In [12]:
df["Merchant"]= df["Details"].str.replace("Paidto","",regex=False).str.replace("Receivedfrom","",regex=False).str.strip().str.title()
print(df["Merchant"])

0             Jayaramapb
1      Shivanna Shivanna
2            Parashurama
3               Rameshmr
4          Mc Nalinakshi
             ...        
508          Amazonindia
509        Mc Nalinakshi
510               Myntra
511        Mc Nalinakshi
512              Blinkit
Name: Merchant, Length: 513, dtype: object


In [13]:
df["Day"]=df["Date"].dt.day
df["Day_Name"]=df["Date"].dt.day_name()
df["Month"]=df["Date"].dt.month_name()
df["Year"]=df["Date"].dt.year


In [14]:
df["Bank"]=(df["Bank"].str.replace("Paidto","",regex=False).str.replace("Paidby","",regex=False).str.strip())

In [15]:
df.head()

,Date,Time,Details,Amount,UPI_ID,Bank,Transaction_Type,Merchant,Day,Day_Name,Month,Year
0,2026-01-03,07:41AM,PaidtoJAYARAMAPB,30.00,600358945338,StateBankofIndia7202,Paid,Jayaramapb,3,Saturday,January,2026
1,2026-01-03,02:17PM,PaidtoSHIVANNA SHIVANNA,231.00,636962750553,StateBankofIndia7202,Paid,Shivanna Shivanna,3,Saturday,January,2026
2,2026-01-05,07:29AM,PaidtoPARASHURAMA,56.15,600553332781,StateBankofIndia7202,Paid,Parashurama,5,Monday,January,2026
3,2026-01-05,09:45AM,PaidtoRAMESHMR,5.00,600558105853,StateBankofIndia7202,Paid,Rameshmr,5,Monday,January,2026
4,2026-01-05,01:14PM,ReceivedfromMC NALINAKSHI,1100.00,969952655934,StateBankofIndia7202,Received,Mc Nalinakshi,5,Monday,January,2026


In [16]:
print(df["Merchant"].unique())

['Jayaramapb' 'Shivanna Shivanna' 'Parashurama' 'Rameshmr' 'Mc Nalinakshi'
 'Blinkit' 'Bmtcbuska57F1114' 'Smartbazaar Bengalurufr47' 'Ka01Ar2969'
 'Mrajeshwaribai' 'Shaik Saleem' 'Sudhakar Chitikela' 'Ganagaraja'
 'Elwineapenm' 'Govindar' 'Aatreyobhattacharya' 'Sadashiva'
 'Veeralchengappa' 'Zepto' 'Ranjancssoshankaregowdat' 'Ka01Ar2872'
 'Swiggy' 'Zeptomarketplaceprivatelimited' 'Ridafathimakhanum'
 'Ka01Ar1160' 'Zeptonow' 'Greentrends' 'Mslifecaremedicals21'
 'Bmtcbuska57F2179' 'Saniya Sultana' 'Soha' 'Mrssaraswathi Ganesan'
 'Lipikasridhar' 'Manjunathahn' 'Mrchennegowda Kc' 'Ssfoodandbeverage'
 'Mrnishanthgowdacm' 'Nagaraja' 'Zabeeralikm' 'Nitikaagarwal'
 'Kishankumar' 'Erannahm' 'Ka51Al6812' 'Mahendrahk' 'Venkateshg'
 'Abhishekr' 'Mr Harshith R' 'Mabu' 'Kumara' 'Spotifyindialip'
 'Manikantas' 'Manjunatha' 'Vigneshvenkatesh' 'Lafemmebeautysaloon'
 'Kavithastores' 'Syedafroz' 'Chaiculture' 'Muskan' 'S Appu'
 'Abhishekshankinamath' 'Bundltechnologiespvtltd' 'Chandrashekhargowda'
 'Aam

- MERCHANT NORMALIZATION

In [17]:
merchant_categorization={
            "Groceries":{
                    "Blinkit":[r"blinkit"],
                    "Swiggy Instamart":[r"swiggy\s*instamart|instamart"],
                    "Zepto":[r"zepto"]
                    },
            "Transport":{
                    "Transport":[r"Ka0|Bmtc|Bmrcl"]
                    },
            "Food":{
                    r"restaurant|chai|coffee|tea|cafe|resto|bar"
                    "Swiggy":[r"swiggy|bundl"],
                    "Zomato":[r"Zomato"],
                    "Ownly":[r"Ownly"]
            },
}

In [18]:
df["Merchant"]=df["Merchant"].str.isinstance(merchant_categorization)

AttributeError: 'StringMethods' object has no attribute 'isinstance'

In [2]:
# Run this inside a Jupyter Notebook cell to retrieve your executed code history
for i, code in enumerate(In):
    if code.strip():
        print(f"--- [CELL {i}] ---")
        print(code)
        print("\n")

--- [CELL 1] ---
# Run this inside a Jupyter Notebook cell (NOT in a terminal .py script)
for i, code in enumerate(In):
    if code.strip():
        print(f"--- [CELL {i}] ---")
        print(code)
        print("\n")


--- [CELL 2] ---
# Run this inside a Jupyter Notebook cell to retrieve your executed code history
for i, code in enumerate(In):
    if code.strip():
        print(f"--- [CELL {i}] ---")
        print(code)
        print("\n")


